In [1]:
# Importing libraries

import pandas as pd
import numpy as np
import synapseclient
import os
import tarfile
import gzip

In [ ]:
# Accessing synapse client
syn = synapseclient.Synapse()
syn.login(authToken="")

folder_id = "syn51365303" #id for european ancestory

files_to_download = list(syn.getChildren(folder_id)) # gets all files within folder
print(f"Found {len(files_to_download)} files total")

In [ ]:
# Loading previous checkpoints -> needed because it kept stopping running
checkpoint_df = pd.read_csv("significant_snps.csv")

# Drop last protein in case it was partially written when kernel died
last_protein = checkpoint_df['source_protein'].iloc[-1] # searching for last protein
checkpoint_df = checkpoint_df[checkpoint_df['source_protein'] != last_protein] # removing that protein
checkpoint_df.to_csv('significant_snps.csv', index=False) # rewriting csv
print(f"Dropped potentially partial protein: '{last_protein}'")

# See which proteins are already done
proteins_done = checkpoint_df['source_protein'].unique().tolist()

# See which proteins had no SNPs but were processed
no_snps_path = "no_significant_snps.txt"
with open(no_snps_path, 'r') as f: # open no SNPs text file
    proteins_no_snps = set(f.read().splitlines())# reads text file as a set and removes the \n between lines

    
print(f"Already processed with results: {len(proteins_done)}")
print(f"Already processed with no results: {len(proteins_no_snps)}")


Dropped potentially partial protein: 'TNN'
Already processed with results: 2462
Already processed with no results: 279


In [ ]:
output_path = "significant_snps.csv" # where csv file should be saved

for i, item in enumerate(files_to_download):
  
    # Skips proteins that have already been processed
    protein_name = item['name'].split('_')[0] # extracts the protein from the file name (always the first word)
    if protein_name in proteins_done or protein_name in proteins_no_snps:
        continue # Skips
    
    print(f'Processing {i+1}/{len(files_to_download)} : {item["name"]}') # gives a measure of progress
    entity = syn.get(item['id'], downloadLocation="./snp_downloads") # gets the synapse ID and adds it to downloads temporarily
    
    protein_had_results = False  # tracks if anything was written -> if there was no SNPs fitting criteria they will be added to separate file
    
    with tarfile.open(entity.path, "r:*") as tar: # opens folder just downloaded
        for member in tar.getmembers():
            if not member.name.endswith('.gz'):
                continue # Skipping the directories and only accessing the actual data
            with gzip.open(tar.extractfile(member), 'rt') as gz: # Opening each file individually
                df = pd.read_csv(gz, sep="\s+")  # Reading file
                df = df[df['LOG10P'] > 7.3] # Only keeping those with high significance threshold
                df = df[(df['A1FREQ'] > 0.01) & (df['A1FREQ'] < 0.99)] # Removing MAF of 0.01 or less (Chen et al. 2021)
                df['F_stat'] = (df['BETA'] / df['SE']) ** 2
                df = df[df['F_stat'] > 10] # Only keeping those with a strong enough f stat
                if not df.empty:
                    df['source_protein'] = protein_name  # Adding the name of the protein
                    df.to_csv(output_path, mode='a', header=False, index=False) # Adding to csv
                    protein_had_results = True # changing for this protein to show there were 1+ SNPs
    
    # Logging to no results if no
    if protein_had_results == False:
        with open(no_snps_path, 'a') as f:
            f.write(protein_name + '\n')
        print('protein had no SNPs that fit criteria')
    
    os.remove(entity.path) # Removing download to save space

print("Protein extraction complete")

In [ ]:
# Reading in extracted proteins

significant_snps = pd.read_csv('significant_snps.csv')

In [ ]:
# Checking dataset 

significant_snps.head()

,CHROM,GENPOS,ID,ALLELE0,ALLELE1,A1FREQ,INFO,N,TEST,BETA,SE,CHISQ,LOG10P,EXTRA,F_stat,source_protein
0,19,57305781,19:57817149:G:A:imp:v1,G,A,0.017355,0.910356,33995,ADD,-0.161838,0.028529,32.1810,7.85245,NaN,32.181041,A1BG
1,19,57479588,19:57990956:A:G:imp:v1,A,G,0.971101,0.974356,33995,ADD,0.138139,0.021505,41.2610,9.87549,NaN,41.261176,A1BG
2,19,57482181,19:57993549:G:A:imp:v1,G,A,0.971820,0.976832,33995,ADD,0.138876,0.021742,40.7984,9.77271,NaN,40.798428,A1BG
3,19,57527596,19:58038964:T:A:imp:v1,T,A,0.974584,0.990548,33995,ADD,0.143538,0.022700,39.9832,9.59149,NaN,39.982911,A1BG
4,19,57725401,19:58236769:T:C:imp:v1,T,C,0.019264,0.976243,33995,ADD,-0.153474,0.026176,34.3753,8.34251,NaN,34.375560,A1BG
